# Mission 3 — 환자 증상 인식 (최종 제출)

119 신고 대화의 전사 텍스트만 읽고 9개 증상을 예측합니다. 평가 지표는 증상별
F1의 평균(Macro F1)이고, 판정 임계값은 **전 증상 0.5**입니다.

이 노트북은 최종 모델의 **실제 학습 로그**를 담고 있습니다. 아래 출력은
`train_local.py`가 GPU에서 남긴 기록을 그대로 옮긴 것입니다.

## 1. 어떻게 이 구성에 도달했는가

약 45회의 짝비교를 Training 내부 holdout(5,840통화)에서 수행했습니다.
공식 Validation은 선택 과정에서 한 번도 사용하지 않았습니다.

### 효과가 있었던 것

| 방법 | Δ Macro F1 | 검증 방식 |
|---|---:|---|
| **앙상블 (확률 단순 평균)** | **+0.0089** | 크기별 부분집합 150개의 평균 |
| **전체 데이터 학습** | **+0.0010** | 데이터 크기 곡선, 최상단 구간 기울기 |
| **이질 구성 (계열 섞기)** | **+0.0016** | 무작위 조합 200개 중 200개 승 |
| **TF-IDF 10% 혼합** | **+0.0012** | 반분 검증 80회, 91% 승 |

앙상블은 **10개에서 포화**합니다 (10개 0.6462 / 16개 0.6463 / 23개 0.6462).
그래서 멤버를 늘리는 대신 **계열을 섞었습니다**. 같은 6개라도 한 레시피로
채우면 0.6448, 계열마다 하나씩 뽑으면 0.6464입니다. 멤버 단독 품질은
같고(0.6374 vs 0.6381) 상호 불일치만 높습니다(0.0295 → 0.0358).

### 효과가 없었던 것

| 축 | 시도 | 결과 |
|---|---|---|
| 전처리 | 텍스트 정규화 | −0.0003 |
| 모델 | 백본 5종 | 전부 klue/roberta-base 이하 |
| 모델 | 구어체 도메인 적응 (MLM) | −0.0008 (6회) |
| 라벨 | 자기 증류 · 라벨 스무딩 · 증상별 손실 가중치 | 전부 음수 |
| 구조 | 로짓 혼합층 · 조합/개수 보조과제 | −0.0034 ~ −0.0073 |
| 분포 | 희귀 증상 오버샘플링 | −0.0109 |
| 손실 | ASL · Focal · 소프트 macro-F1 | 전부 음수 |
| 결합 | 스태킹 · 증상별 혼합비율 | 전부 음수 |
| 후처리 | 최소 1개 증상 강제 | −0.0008 |

### 측정의 한계를 함께 적습니다

| 각 설정의 실행 수 | 탐지 가능 최소 효과 |
|---:|---:|
| 3 (주로 사용) | 0.0044 |
| 40 | 0.0012 |

잡음은 seed(표준편차 0.0023)와 평가(0.0015)에서 옵니다. **"효과 없음" 중
상당수는 "이 설계로 못 쟀다"입니다.** TF-IDF와 이질 구성만 별도 검증을
거쳤기에 확정이라 적었습니다.

## 2. 데이터 분석이 알려준 것

### 모델은 전사에서 읽을 수 있는 것을 거의 다 읽습니다

놓친 정답(FN)과 잘못 짚은 것(FP)에 전사 근거가 있는지 셌습니다.

| 증상 | FN 수 | FN에 표현 있음 | FP 수 | FP에 표현 있음 |
|---|---:|---:|---:|---:|
| 구토 | 404 | **1.5%** | 280 | 50.4% |
| 두통 | 309 | 10.4% | 240 | 56.2% |
| 호흡곤란 | 210 | 33.3% | 313 | **86.9%** |

**양쪽 다 어노테이션 행동입니다.** 구토는 놓친 404건 중 6건에만 구토 표현이
있습니다 — 읽고도 못 맞힌 게 아니라 읽을 게 없습니다. 반대로 호흡곤란 FP의
87%에 "숨"이 나옵니다.

### 오심만 질적으로 다릅니다

표현이 있는 통화 안에서 라벨 유무를 구분할 수 있는가(조건부 AUC):

| 열상 | 고열 | 복통 | 호흡곤란 | 전신쇠약 | 두통 | 어지러움 | 구토 | **오심** |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0.980 | 0.940 | 0.897 | 0.871 | 0.842 | 0.777 | 0.774 | 0.726 | **0.567** |

오심 전체 AUC 0.777은 거의 전부 "이게 구토·어지러움 통화인가"를 맞히는 데서
나옵니다. 그 안에서 오심이 추가로 붙었는지는 **동전던지기**입니다.
"메스꺼워요"가 전사에 또박또박 있어도 오심 라벨은 **63%만** 붙습니다.

오심이 붙은 통화와 안 붙은 통화를 가르는 단어를 가정 없이 채굴해도,
올라오는 것은 `어지럽고`(공동출현 재확인)와 `노원구`·`위치` 같은 주소·상투어
입니다 — 신호가 없을 때 나오는 모습입니다.

### 임계값 0.5 고정의 비용은 +0.0062

증상별 임계값을 자유롭게 골랐다면 0.6441 → 0.6503입니다. 단조 보정은 순위를
못 바꾸므로 **확률 보정으로 얻을 수 있는 것도 이게 전부**입니다.

## 3. 모델 코드

In [1]:
"""Fine-tune a Korean encoder for the nine symptoms, on this machine's GPU.

Deliberately independent of ver4: weights start from the public pretrained
checkpoint, the split is our own Training-internal grouped split, and official
Validation is never read. That keeps every result here free of the anchoring
that ver5-ver8 ran into.

Decisions are scored at the mandated 0.5 threshold for every symptom.
"""
from __future__ import annotations

import argparse
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup,
                          get_linear_schedule_with_warmup)

sys.path.insert(0, str(Path(__file__).parent))
from common_local import TARGETS, TRAIN_DIR, load, macro_f1, split          # noqa: E402
from normalize import VARIANTS, normalize                                    # noqa: E402

OUT = Path(__file__).resolve().parents[3] / "results" / "mission3" / "local"


class Calls(Dataset):
    def __init__(self, texts, Y=None, aux=None):
        self.texts, self.Y, self.aux = texts, Y, aux

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        y = self.Y[i] if self.Y is not None else 0
        return self.texts[i], y, (self.aux[i] if self.aux is not None else -100)


class Collate:
    def __init__(self, tokenizer, max_length):
        self.tok, self.max_length = tokenizer, max_length

    def __call__(self, batch):
        enc = self.tok([b[0] for b in batch], padding=True, truncation=True,
                       max_length=self.max_length, return_tensors="pt")
        enc["labels"] = torch.tensor(np.stack([b[1] for b in batch]), dtype=torch.float32)
        enc["aux"] = torch.tensor([b[2] for b in batch], dtype=torch.long)
        return enc


class Classifier(nn.Module):
    """Mean pooling, or one attention query per symptom (LAAT / PLM-ICD style).

    Mean pooling forces all nine symptoms to share a single sentence vector.
    Label attention lets each symptom weight the tokens itself, so "열상" can
    look at "피가 나요" while "오심" looks somewhere else entirely -- which is
    the pooling ver4 used and this pipeline did not.
    """

    def __init__(self, name, n_labels, dropout=0.1, head="mean", attn_dim=256,
                 label_corr=False, aux_dim=0):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        self.drop = nn.Dropout(dropout)
        self.head_kind = head
        hidden = self.encoder.config.hidden_size
        # Nine independent sigmoids ignore that the labels travel together:
        # 오심 shows up beside 구토 and 어지러움 far more often than chance.
        # A zero-initialised residual mixer starts as the identity, so the run
        # only diverges from the plain head if the correlation actually pays.
        self.corr = None
        if label_corr:
            self.corr = nn.Linear(n_labels, n_labels)
            nn.init.zeros_(self.corr.weight)
            nn.init.zeros_(self.corr.bias)
        # Auxiliary softmax over whole label combinations. Only 188 of the 511
        # possible sets ever occur, and 어지러움+오심 is the ninth most common --
        # structure nine independent sigmoids cannot represent. The head is
        # dropped at inference, so it only shapes the encoder.
        self.aux = nn.Linear(hidden, aux_dim) if aux_dim else None
        if head in ("mean", "cls", "max"):
            self.head = nn.Linear(hidden, n_labels)
        else:
            self.project = nn.Linear(hidden, attn_dim)
            self.queries = nn.Linear(attn_dim, n_labels, bias=False)
            self.out_weight = nn.Parameter(torch.empty(n_labels, hidden))
            self.out_bias = nn.Parameter(torch.zeros(n_labels))
            nn.init.xavier_uniform_(self.out_weight)

    def forward(self, input_ids, attention_mask, **kw):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
        if self.head_kind in ("mean", "cls", "max"):
            # Three ways to collapse the tokens into one vector. They score
            # about the same on their own, but they are wrong about different
            # calls -- which is what an ensemble wants. "cls" reads the
            # position the encoder was pretrained to summarise with; "max"
            # takes the strongest activation per dimension, so a single
            # decisive phrase can carry the whole call.
            if self.head_kind == "cls":
                pooled = hidden[:, 0]
            elif self.head_kind == "max":
                pooled = hidden.masked_fill(mask == 0, float("-inf")).max(1).values
            logits = self.head(self.drop(pooled))
        else:
            # scores: (batch, tokens, labels); padding must never win attention.
            scores = self.queries(torch.tanh(self.project(hidden)))
            scores = scores.masked_fill(attention_mask.unsqueeze(-1) == 0, float("-inf"))
            weights = torch.softmax(scores.float(), dim=1).to(hidden.dtype)
            # per-label context: (batch, labels, hidden)
            context = torch.einsum("btl,bth->blh", weights, hidden)
            context = self.drop(context)
            logits = (context * self.out_weight).sum(-1) + self.out_bias
        if self.corr is not None:
            logits = logits + self.corr(logits)
        return (logits, self.aux(self.drop(pooled))) if self.aux is not None else logits


def pos_weight(Y, device, scheme="sqrt5"):
    """How hard to push each symptom's probability toward the positive side.

    With the decision threshold frozen at 0.5, this is the main remaining way
    to place a symptom's probabilities relative to the cut. ver4 could instead
    move the cut per symptom (오심 sat at 0.411); that option is gone, so the
    weighting has to carry it. "sqrt5" is ver4's inherited setting.
    """
    pos = Y.sum(axis=0).astype(np.float64)
    ratio = np.maximum((len(Y) - pos) / np.maximum(pos, 1.0), 1.0)
    weights = {
        "none": np.ones_like(ratio),
        "sqrt3": np.clip(np.sqrt(ratio), 1.0, 3.0),
        "sqrt5": np.clip(np.sqrt(ratio), 1.0, 5.0),
        "sqrt8": np.clip(np.sqrt(ratio), 1.0, 8.0),
        "sqrt": np.sqrt(ratio),
        # "none" (1.0) scored 0.619 and "sqrt" (1.8-3.0) scored 0.638, so the
        # useful direction is stronger weighting. These walk further along it.
        # Soft targets compress the outputs toward the middle, and with sqrt
        # already inflating them the result crosses 0.5 far too often (9,583
        # predictions against 8,305 true). These sit below sqrt so a distilled
        # run can be pulled back. The formula stays the same for all nine
        # symptoms -- this is global class weighting, not a per-symptom 

... (중략) ...

mble pool.
        np.save(OUT / f"oof_{tag}_k{args.folds}f{args.fold}.npy", best_probs)
        np.save(OUT / f"oofidx_{tag}_k{args.folds}f{args.fold}.npy", te)
    elif args.full_train:
        # No honest holdout exists here, so nothing goes into the pool that the
        # ensemble analysis reads from.
        np.save(OUT / f"insample_{tag}.npy", best_probs)
    else:
        np.save(OUT / f"probs_{tag}.npy", best_probs)
        np.save(OUT / "holdout_Y.npy", Y[te])
    if args.save_model:
        ckpt_dir = OUT / "submission_models"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                    "model_name": args.arch_name or args.model, "head": args.head,
                    "max_length": args.max_length, "dropout": args.dropout,
                    "target_symptoms": list(TARGETS), "threshold": 0.5,
                    "normalize": args.normalize,
                    # The encoder's own config travels with the weights, so
                    # inference can build the architecture without fetching
                    # 440MB of pretrained weights it immediately overwrites.
                    "encoder_config": model.encoder.config.to_dict(),
                    "trained_on": int(len(tr)), "epochs": args.epochs,
                    "lr": args.lr, "seed": args.seed},
                   ckpt_dir / f"{tag}.ckpt")
        tok.save_pretrained(ckpt_dir / "tokenizer")
        log(f"  가중치 저장: submission_models/{tag}.ckpt")
    (OUT / f"history_{tag}.json").write_text(
        json.dumps({"args": vars(args), "history": history, "best": best},
                   ensure_ascii=False, indent=2), encoding="utf-8")
    log(f"  최고 epoch {best['epoch']}: {best['best_macro_f1']:.5f} "
        f"(그대로 {best['macro_f1']:.5f} / 강제 {best['macro_f1_forced']:.5f} 중 높은 쪽)")
    return best


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--model", default="klue/roberta-base")
    p.add_argument("--normalize", default="raw", choices=list(VARIANTS))
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--eval_batch_size", type=int, default=64)
    p.add_argument("--accum", type=int, default=2)
    p.add_argument("--lr", type=float, default=3e-5)
    p.add_argument("--max_length", type=int, default=512)
    p.add_argument("--dropout", type=float, default=0.1)
    p.add_argument("--head", default="mean",
                   choices=("mean", "cls", "max", "label_attention"))
    p.add_argument("--pos_weight", default="sqrt5",
                   choices=("none", "pow025", "pow035", "sqrt3", "sqrt5", "sqrt8", "sqrt",
                            "pow065", "pow080", "full"))
    p.add_argument("--loss", default="bce", choices=("bce", "asl", "focal"))
    p.add_argument("--gamma_neg", type=float, default=2.0, help="ASL/focal focusing strength.")
    p.add_argument("--ema", type=float, default=0.0, help="EMA decay; 0 disables.")
    p.add_argument("--train_frac", type=float, default=1.0, help="Fraction of the train split to use.")
    p.add_argument("--bag_seed", type=int, default=0, help="Which subsample, when train_frac < 1.")
    p.add_argument("--soft_targets", default="",
                   help="(29200, 9) teacher probabilities aligned with load() order.")
    p.add_argument("--soft_alpha", type=float, default=0.3,
                   help="Mixing weight: target = (1-a)*hard + a*soft.")
    p.add_argument("--label_weight", default="none",
                   help="none | invfreq | custom:w1,...,w9 -- per-symptom loss scale.")
    p.add_argument("--scheduler", default="linear", choices=("linear", "cosine"))
    p.add_argument("--warmup", type=float, default=0.1,
                   help="Fraction of steps spent warming up.")
    p.add_argument("--softf1", type=float, default=0.0,
                   help="Weight of a differentiable macro-F1 term added to BCE; 0 disables.")
    p.add_argument("--oversample", default="none", choices=("none", "log", "sqrt", "full"),
                   help="Draw calls carrying rare symptoms more often.")
    p.add_argument("--rdrop", type=float, default=0.0,
                   help="R-Drop strength; 0 disables. Costs a second forward pass.")
    p.add_argument("--aux", default="none",
                   help="none | count | combo:K -- training-only auxiliary head.")
    p.add_argument("--aux_weight", type=float, default=0.3)
    p.add_argument("--label_corr", action="store_true",
                   help="Zero-initialised residual mixer over the nine logits.")
    p.add_argument("--fold", type=int, default=-1,
                   help="With --folds k, train on all but this fold of the train split.")
    p.add_argument("--folds", type=int, default=0, help="Number of OOF folds; 0 disables.")
    p.add_argument("--llrd", type=float, default=1.0,
                   help="Layer-wise LR decay; 1.0 keeps every layer at --lr.")
    p.add_argument("--full_train", action="store_true",
                   help="Train on all 29,200 calls; no holdout is left to score on.")
    p.add_argument("--arch_name", default="",
                   help="Public checkpoint name to record in the saved file. A domain-adapted run trains from a local directory, and that path will not exist on the grading machine -- the weights live in the state_dict, so only the architecture name needs to be portable.")
    p.add_argument("--save_model", action="store_true", help="Write a submission checkpoint.")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--grad_checkpoint", action="store_true")
    p.add_argument("--tag_suffix", default="", help="Keeps runs that differ only by epochs from overwriting each other.")
    p.add_argument("--log", default=str(OUT / "train_log.md"))
    args = p.parse_args()

    OUT.mkdir(parents=True, exist_ok=True)
    logfile = Path(args.log)

    def log(line):
        print(line, flush=True)
        with logfile.open("a", encoding="utf-8") as handle:
            handle.write(line + "\n")

    log(f"\n### {time.strftime('%H:%M:%S')} {args.model} / {args.normalize} / seed {args.seed}")

    # 8GB is tight for the large encoder. Rather than lose a queued run to an
    # out-of-memory crash, halve the per-step batch and keep the effective batch
    # the same by doubling accumulation, then try again.
    while True:
        try:
            run(args, log)
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect()          # drop the traceback's hold on the dead model
            torch.cuda.empty_cache()
            if args.batch_size <= 1:
                log("  VRAM 부족: batch 1에서도 실패해 이 설정은 건너뜁니다.")
                raise
            args.batch_size //= 2
            args.accum *= 2
            args.eval_batch_size = max(1, args.eval_batch_size // 2)
            log(f"  VRAM 부족 → batch {args.batch_size} / accum {args.accum} 로 재시도")

## 4. 최종 학습 설정 — 한 레시피가 아니라 일곱 계열

앙상블은 10개에서 포화하므로(10개 0.6462 / 16개 0.6463 / 23개 0.6462) 멤버를
늘리는 길은 닫혀 있습니다. 남은 지렛대는 **계열 수**였습니다.

| 계열 | 멤버 | 무엇이 다른가 |
|---|---:|---|
| base mean lr2e-5 | 2 | 기준 레시피 |
| base attn lr2e-5 | 2 | 증상별 attention 헤드 |
| base mean lr3e-5 | 1 | 학습률 |
| base attn lr3e-5 | 1 | 학습률 + 헤드 |
| 정규화 텍스트 | 2 | 동의어 통합 + 필러 제거 |
| 도메인 적응 | 2 | 전사 29,200개로 MLM 3 epoch 후 fine-tune |
| CLS 풀링 | 1 | `[CLS]` 토큰 사용 |

공통: 3 epoch · `pos_weight = clip(sqrt(음성/양성), 1, 5)` · batch 16 × accum 2 ·
AdamW · bf16 · Training 29,200개 전부 · **확률 단순 평균 + TF-IDF 10%** ·
**판정 임계값 전 증상 0.5** · EMA 없음 · 후처리 없음.

**전체 데이터로 학습하므로 로컬 검증 세트가 없습니다.** 아래 로그의 Macro F1은
학습에 포함된 통화로 잰 값이라 일반화 성능이 아니며, 수렴 확인용입니다.
구성은 모두 holdout 실험에서 먼저 확정했습니다 — 같은 계열 구성을 holdout
학습 모델로 조립한 대리값이 **0.64666**이고, 한 레시피로만 채운 구성은
0.64484입니다.

## 5. 실제 학습 로그

In [1]:
# 아래는 학습 중 GPU가 남긴 실제 출력입니다. 제출 모델은 두 번의
# 학습에서 나옵니다 — 기존 레시피 4개와 계열을 넓힌 7개.
for f in ('final_train_log.md', 'final_diverse_log.md'):
    print(open(f'results/mission3/local/{f}', encoding='utf-8').read())


### 20:55:08 klue/roberta-base / raw / seed 42
모델 klue/roberta-base | 전처리 raw | 손실 bce | 학습 29,200 / 자체검증 5,840
  epoch 1: loss 0.5036 | Macro F1 0.65159(학습에 포함됨·참고용) | 최소1개 강제 0.65084 | 빈 예측 189 | 5.7분
  epoch 2: loss 0.4145 | Macro F1 0.67587(학습에 포함됨·참고용) | 최소1개 강제 0.67540 | 빈 예측 103 | 5.7분
  epoch 3: loss 0.3827 | Macro F1 0.70337(학습에 포함됨·참고용) | 최소1개 강제 0.70245 | 빈 예측 86 | 5.7분
  가중치 저장: submission_models/roberta-base_raw_s42_FINAL.ckpt
  최고 epoch 3: 0.70337 (그대로 0.70337 / 강제 0.70245 중 높은 쪽)

### 21:18:32 klue/roberta-base / raw / seed 1
모델 klue/roberta-base | 전처리 raw | 손실 bce | 학습 29,200 / 자체검증 5,840
  epoch 1: loss 0.5033 | Macro F1 0.62910(학습에 포함됨·참고용) | 최소1개 강제 0.62701 | 빈 예측 162 | 5.7분
  epoch 2: loss 0.4147 | Macro F1 0.67753(학습에 포함됨·참고용) | 최소1개 강제 0.67703 | 빈 예측 124 | 5.7분
  epoch 3: loss 0.3848 | Macro F1 0.69742(학습에 포함됨·참고용) | 최소1개 강제 0.69760 | 빈 예측 75 | 5.7분
  가중치 저장: submission_models/roberta-base_raw_s1_FINAL.ckpt
  최고 epoch 3: 0.69760 (그대로 0.69742 / 강제 0.69760 중 높은 쪽)


## 6. 추론

제출 규격(출제문제 11쪽)에 맞춘 한 줄 실행입니다.

```bash
python inference.py --audio_dir {wav folder} --label_dir {json folder} \
    --ckpt_path {checkpoint folder} --output ./outputs/mission3.csv
```

Mission 3은 음성을 읽지 않지만 `--audio_dir`를 받습니다. `--ckpt_path`에
폴더를 주면 안의 모든 체크포인트를 앙상블합니다.

In [1]:
"""Mission 3 submission inference.

Required interface (출제문제 11쪽):

    python inference.py --audio_dir {wav folder} --label_dir {json folder} \
        --ckpt_path {checkpoint file} --output ./outputs/mission3.csv

One command produces the CSV. Mission 3 reads no audio, but --audio_dir is
accepted because the organisers' harness passes it.

What this does:
  * reads only utterances[].text from each annotation JSON
  * averages the probabilities of every checkpoint in the ensemble
  * calls a symptom present when its probability is >= 0.5 -- the same
    threshold for all nine, as the organisers require
  * writes `label file name,symptom` with the symptom list as a string

Pass a directory to --ckpt_path to ensemble every .ckpt inside it, or a single
.ckpt file for one model. Both work; the submitted ensemble is a directory.

If a `tfidf.pkl` sits beside the checkpoints it is blended in at 10%, which
is worth +0.0011 Macro F1 and +0.0026 AP on the holdout. See tfidf_model.py
for the split-half validation behind that weight.
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

TARGETS = ("고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란")
THRESHOLD = 0.5
UTTERANCE_KEYS = ("utterances", "utterences")


class Classifier(nn.Module):
    """Must match the architecture the checkpoints were trained with."""

    def __init__(self, name, n_labels, dropout=0.1, head="mean", attn_dim=256,
                 encoder_config=None):
        super().__init__()
        # from_pretrained downloads the published weights and load_state_dict
        # then replaces every one of them. Building from the stored config
        # skips that: no network, and eleven checkpoints load in seconds.
        if encoder_config:
            cfg = dict(encoder_config)
            self.encoder = AutoModel.from_config(
                AutoConfig.for_model(cfg.pop("model_type"), **cfg))
        else:
            self.encoder = AutoModel.from_pretrained(name)
        self.drop = nn.Dropout(dropout)
        self.head_kind = head
        hidden = self.encoder.config.hidden_size
        if head in ("mean", "cls", "max"):
            self.head = nn.Linear(hidden, n_labels)
        else:
            self.project = nn.Linear(hidden, attn_dim)
            self.queries = nn.Linear(attn_dim, n_labels, bias=False)
            self.out_weight = nn.Parameter(torch.empty(n_labels, hidden))
            self.out_bias = nn.Parameter(torch.zeros(n_labels))
            nn.init.xavier_uniform_(self.out_weight)

    def forward(self, input_ids, attention_mask, **kw):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        if self.head_kind in ("mean", "cls", "max"):
            # Three ways to collapse tokens into one vector. They must match
            # train_local.py exactly -- a head the trainer knows and this file
            # does not fails at load_state_dict, which is how "cls" was caught.
            if self.head_kind == "cls":
                pooled = hidden[:, 0]
            elif self.head_kind == "max":
                pooled = hidden.masked_fill(mask == 0, float("-inf")).max(1).values
            else:
                pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
            return self.head(self.drop(pooled))
        scores = self.queries(torch.tanh(self.project(hidden)))
        scores = scores.masked_fill(attention_mask.unsqueeze(-1) == 0, float("-inf"))
        weights = torch.softmax(scores.float(), dim=1).to(hidden.dtype)
        context = self.drop(torch.einsum("btl,bth->blh", weights, hidden))
        return (context * self.out_weight).sum(-1) + self.out_bias


def dialogue_text(annotation):
    """Only the transcript. speaker, startAt, endAt and the rest stay unread."""
    utterances = []
    for key in UTTERANCE_KEYS:
        value = annotation.get(key)
        if isinstance(value, list):
            utterances = value
            break
    said = [u["text"].strip() for u in utterances
            if isinstance(u, dict) and isinstance(u.get("text"), str) and u["text"].strip()]
    return "\n".join(said)


def read_inputs(label_dir: Path):
    """Every annotation JSON under label_dir, and nothing else.

    rglob("*.json") happily returns any J

... (중략) ...

ormalize
        texts = [normalize(t, **VARIANTS[variant]) for t in texts]
    # One tokenizer folder is shared by the whole ensemble, which is fine only
    # while every member uses the same vocabulary. A heterogeneous ensemble can
    # break that quietly -- wrong token ids produce plausible-looking garbage
    # rather than an error -- so the vocabulary size is checked against the
    # checkpoint's own embedding matrix before the shared copy is trusted.
    vocab_rows = blob["state_dict"]["encoder.embeddings.word_embeddings.weight"].shape[0]
    tokenizer_dir = ckpt_path.parent / "tokenizer"
    tok = None
    if tokenizer_dir.exists():
        shared = AutoTokenizer.from_pretrained(str(tokenizer_dir), use_fast=True)
        if len(shared) <= vocab_rows:
            tok = shared
        else:
            print(f"    공유 토크나이저 불일치 ({len(shared)} vs {vocab_rows}) — "
                  f"{blob['model_name']}에서 직접 불러옵니다", flush=True)
    if tok is None:
        tok = AutoTokenizer.from_pretrained(blob["model_name"], use_fast=True)
    model = Classifier(blob["model_name"], len(blob["target_symptoms"]),
                       blob.get("dropout", 0.1), blob.get("head", "mean"),
                       encoder_config=blob.get("encoder_config"))
    model.load_state_dict(blob["state_dict"])
    model.to(device).eval()
    amp = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else None

    out = []
    for start in range(0, len(texts), batch_size):
        batch = tok(texts[start:start + batch_size], padding=True, truncation=True,
                    max_length=blob.get("max_length", 512), return_tensors="pt")
        batch = {k: v.to(device) for k, v in batch.items()}
        if amp is not None:
            with torch.autocast("cuda", dtype=amp):
                logits = model(**batch)
        else:
            logits = model(**batch)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(out)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--audio_dir", default="", help="미사용 (대회 인터페이스 호환).")
    parser.add_argument("--label_dir", required=True)
    parser.add_argument("--ckpt_path", required=True,
                        help=".ckpt 파일 또는 앙상블 체크포인트가 든 폴더.")
    parser.add_argument("--output", default="./outputs/mission3.csv")
    parser.add_argument("--tfidf_path", default="",
                        help="기본값은 체크포인트 폴더의 tfidf.pkl.")
    parser.add_argument("--tfidf_weight", type=float, default=0.10,
                        help="0이면 TF-IDF를 섞지 않습니다.")
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--device", choices=("cuda", "cpu"))
    args = parser.parse_args()

    device = torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))
    names, texts = read_inputs(Path(args.label_dir))
    print(f"통화 {len(names):,}개 · 장치 {device}", flush=True)

    ckpt = Path(args.ckpt_path)
    checkpoints = sorted(ckpt.glob("*.ckpt")) if ckpt.is_dir() else [ckpt]
    if not checkpoints:
        raise SystemExit(f"체크포인트를 찾지 못했습니다: {ckpt}")

    total = None
    for i, path in enumerate(checkpoints, 1):
        print(f"  [{i}/{len(checkpoints)}] {path.name}", flush=True)
        p = probabilities(path, texts, device, args.batch_size)
        total = p if total is None else total + p
    probs = total / len(checkpoints)

    tfidf = Path(args.tfidf_path) if args.tfidf_path else \
        (ckpt if ckpt.is_dir() else ckpt.parent) / "tfidf.pkl"
    if tfidf.exists() and args.tfidf_weight > 0:
        sys.path.insert(0, str(Path(__file__).parent))
        import tfidf_model
        print(f"  TF-IDF {tfidf.name} 혼합 {args.tfidf_weight:g}", flush=True)
        probs = ((1 - args.tfidf_weight) * probs
                 + args.tfidf_weight * tfidf_model.predict(tfidf_model.load(tfidf), texts))

    output = Path(args.output)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.writer(handle)
        writer.writerow(["label file name", "symptom"])
        for name, row in zip(names, probs):
            found = [s for s, value in zip(TARGETS, row) if value >= THRESHOLD]
            writer.writerow([name, str(found)])
    print(f"저장 완료: {output}  (모델 {len(checkpoints)}개 평균, 임계값 {THRESHOLD})")


if __name__ == "__main__":
    main()